# GenAI Contract Demo

This notebook shows a practical FastAPI endpoint for a GenAI-style document summary API.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List

app = FastAPI(title="GenAI Contract Demo")

In [ ]:
class SummarizeRequest(BaseModel):
    question: str = Field(..., min_length=5)
    document_id: int = Field(..., gt=0)
    max_tokens: int = Field(default=200, ge=50, le=1000)
    temperature: float = Field(default=0.2, ge=0.0, le=1.0)

class SourceItem(BaseModel):
    document_id: int
    chunk_id: int

class SummarizeResponse(BaseModel):
    status: str
    answer: str
    model: str
    sources: List[SourceItem]

In [ ]:
DOCUMENT_STORE = {
    10: "Annual report content",
    11: "Customer review digest"
}

@app.post("/summaries", response_model=SummarizeResponse)
def summarize(payload: SummarizeRequest):
    if payload.document_id not in DOCUMENT_STORE:
        raise HTTPException(status_code=404, detail="Document not found")

    return {
        "status": "success",
        "answer": f"Summary for document {payload.document_id} based on: {payload.question}",
        "model": "demo-llm",
        "sources": [
            {"document_id": payload.document_id, "chunk_id": 1}
        ]
    }

Example request body:

```json
{
  "question": "Summarize the annual report",
  "document_id": 10,
  "max_tokens": 250,
  "temperature": 0.2
}
```

This pattern is useful for RAG, summarization, and document QA APIs.